# Librerias

In [37]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [38]:
CLEAN_DIR = Path("../data_clean")
pd.set_option("display.max_columns", 50)

# Funciones

In [39]:
def max_consecutive_run(bool_series: pd.Series) -> int:
    """
    Longitud de la racha 'True' más larga dentro de una serie booleana
    ya ordenada cronológicamente (ej. días secos consecutivos).
    """
    if bool_series.empty:
        return 0
    groups = (~bool_series).cumsum()
    run_lengths = bool_series.groupby(groups).cumsum()
    return int(run_lengths.max())


def dry_spell_features(df: pd.DataFrame, precip_col: str, dry_threshold_mm: float = 1.0,
                        date_col: str = "fecha", muni_col: str = "municipio") -> pd.DataFrame:
    """
    Calcula, por municipio-año:
      - dias_secos_consecutivos_max: racha seca más larga del año
      - dias_secos_totales: número total de días secos en el año
      - dias_lluviosos_totales: número de días con lluvia >= threshold
      - precip_dias_lluviosos_promedio: intensidad media en días con lluvia (mm/día lluvioso)
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df["anio"] = df[date_col].dt.year
    df = df.sort_values([muni_col, date_col])
    df["es_seco"] = df[precip_col] < dry_threshold_mm

    out = []
    for (muni, anio), g in df.groupby([muni_col, "anio"]):
        g = g.sort_values(date_col)
        dias_lluviosos = (~g["es_seco"]).sum()
        precip_dias_lluviosos = g.loc[~g["es_seco"], precip_col]
        out.append({
            muni_col: muni,
            "anio": anio,
            "dias_secos_consecutivos_max": max_consecutive_run(g["es_seco"]),
            "dias_secos_totales": int(g["es_seco"].sum()),
            "dias_lluviosos_totales": int(dias_lluviosos),
            "precip_dia_lluvioso_promedio": precip_dias_lluviosos.mean() if dias_lluviosos > 0 else 0.0,
        })
    return pd.DataFrame(out)


def rolling_window_sum(df: pd.DataFrame, value_col: str, window_days: int,
                        window_start_month: int, window_end_month: int,
                        date_col: str = "fecha", muni_col: str = "municipio",
                        out_col_name: str = None) -> pd.DataFrame:
    """
    Suma de una variable diaria dentro de una ventana fenológica fija (ej. floración: marzo-abril).
    Ojo: esto NO es una ventana móvil de N días respecto a "hoy" (no aplica para features anuales
    retrospectivas); es la suma de la variable dentro del rango de meses indicado, por año.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df["anio"] = df[date_col].dt.year
    mask = df[date_col].dt.month.between(window_start_month, window_end_month)
    sub = df[mask]
    col_name = out_col_name or f"{value_col}_ventana_{window_start_month}_{window_end_month}"
    agg = sub.groupby([muni_col, "anio"])[value_col].sum().reset_index()
    agg = agg.rename(columns={value_col: col_name})
    return agg


def historical_anomaly(annual_df: pd.DataFrame, value_col: str,
                        muni_col: str = "municipio", year_col: str = "anio") -> pd.DataFrame:
    """
    Anomalía respecto a la media histórica DEL MISMO MUNICIPIO (no del país/departamento):
      anomalia = (valor_anio - media_historica_municipio) / desv_estandar_historica_municipio
    Importante: usar solo con la serie histórica disponible hasta ese punto en producción real,
    para evitar fuga de información (data leakage) al entrenar con años futuros. Para el
    prototipo, se calcula con toda la serie disponible y se documenta como limitación conocida.
    """
    annual_df = annual_df.copy()
    stats = annual_df.groupby(muni_col)[value_col].agg(["mean", "std"]).reset_index()
    stats = stats.rename(columns={"mean": f"{value_col}_media_hist", "std": f"{value_col}_std_hist"})
    merged = annual_df.merge(stats, on=muni_col, how="left")
    merged[f"{value_col}_anomalia"] = (
        (merged[value_col] - merged[f"{value_col}_media_hist"]) / merged[f"{value_col}_std_hist"]
    )
    return merged


def growing_degree_days(df: pd.DataFrame, tmax_col: str, tmin_col: str,
                         base_temp: float = 10.0, date_col: str = "fecha",
                         muni_col: str = "municipio") -> pd.DataFrame:
    """
    Growing Degree Days (GDD) anual: acumulación de calor disponible para desarrollo del cultivo.
    GDD_dia = max(0, (Tmax+Tmin)/2 - T_base). T_base=10°C es un valor típico usado en literatura
    de café; ajustar si tienes evidencia agronómica específica para Boyacá.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df["anio"] = df[date_col].dt.year
    df["gdd_dia"] = ((df[tmax_col] + df[tmin_col]) / 2 - base_temp).clip(lower=0)
    return df.groupby([muni_col, "anio"])["gdd_dia"].sum().reset_index().rename(
        columns={"gdd_dia": "gdd_anual"}
    )


def frost_and_heat_days(df: pd.DataFrame, tmin_col: str, tmax_col: str,
                         frost_threshold: float = 8.0, heat_threshold: float = 30.0,
                         date_col: str = "fecha", muni_col: str = "municipio") -> pd.DataFrame:
    """
    Conteo anual de:
      - dias_friofrio: Tmin por debajo del umbral de estrés por frío (daño foliar/floración)
      - dias_calor_extremo: Tmax por encima del umbral de estrés por calor
    Los umbrales por defecto (8°C / 30°C) son orientativos para café de montaña; conviene
    validarlos con literatura agronómica local antes del informe final.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df["anio"] = df[date_col].dt.year
    df["es_dia_frio"] = df[tmin_col] < frost_threshold
    df["es_dia_calor"] = df[tmax_col] > heat_threshold
    return df.groupby([muni_col, "anio"]).agg(
        dias_frio=("es_dia_frio", "sum"),
        dias_calor_extremo=("es_dia_calor", "sum"),
    ).reset_index()

def detect_flowering_events(df: pd.DataFrame, precip_col: str = "precipitation_chirps",
                             trigger_mm: float = 10.0, trigger_window_days: int = 3,
                             min_dry_days_before: int = 10, dry_threshold_mm: float = 1.0,
                             date_col: str = "fecha", muni_col: str = "municipio",
                             max_eventos_por_anio: int = 2) -> pd.DataFrame:
    """
    Detecta eventos de floración del café a partir de precipitación diaria, replicando
    el mecanismo agronómico documentado por Cenicafé: la floración se dispara por una
    lluvia significativa (~10mm en pocos días) que rompe un período seco previo.
    Referencia: Cenicafé documentó un caso de floración del 78% en respuesta a una
    lluvia de 10.1mm ocurrida 8 días después de iniciado un período seco.

    A diferencia de una ventana fenológica fija por calendario, este método:
      - Se adapta a cada municipio-año (no asume el mismo mes en todos lados)
      - Captura que la altitud modula el momento de floración (municipios más altos
        suelen florecer más tarde, según el mismo estudio de Cenicafé)
      - Permite hasta 2 eventos por año (floración principal + "mitaca")

    Devuelve un dataframe con: municipio, anio, evento (1=principal, 2=mitaca),
    fecha_floracion, fecha_cosecha_estimada (floración + 8 meses, según literatura).
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values([muni_col, date_col])

    eventos = []
    for muni, g in df.groupby(muni_col):
        g = g.sort_values(date_col).reset_index(drop=True)
        lluvia_3d = g[precip_col].rolling(trigger_window_days).sum()
        dry_prev = (g[precip_col] < dry_threshold_mm).rolling(min_dry_days_before).sum()

        candidatos = g[(lluvia_3d >= trigger_mm) & (dry_prev >= min_dry_days_before * 0.8)].copy()
        if candidatos.empty:
            continue
        candidatos["anio"] = candidatos[date_col].dt.year

        # Evita eventos duplicados muy cercanos en el tiempo (ej. lluvias de la misma tormenta)
        candidatos = candidatos.sort_values(date_col)
        candidatos["dias_desde_anterior"] = candidatos[date_col].diff().dt.days.fillna(999)
        candidatos = candidatos[candidatos["dias_desde_anterior"] > 30]

        for anio, grupo_anio in candidatos.groupby("anio"):
            grupo_anio = grupo_anio.nlargest(max_eventos_por_anio, precip_col)
            for i, (_, fila) in enumerate(grupo_anio.sort_values(date_col).iterrows(), start=1):
                eventos.append({
                    muni_col: muni,
                    "anio": anio,
                    "evento": i,
                    "fecha_floracion": fila[date_col],
                    "fecha_cosecha_estimada": fila[date_col] + pd.DateOffset(months=8),
                })

    return pd.DataFrame(eventos)


def phenology_window_features(daily_df: pd.DataFrame, eventos: pd.DataFrame, value_col: str,
                               dias_ventana: int = 20, date_col: str = "fecha",
                               muni_col: str = "municipio",
                               agg: str = "sum") -> pd.DataFrame:
    """
    Agrega una variable diaria (ej. precipitación, o NDVI si se remuestrea a diario)
    dentro de una ventana de +/- `dias_ventana` alrededor de cada evento de floración
    y de su cosecha estimada (floración + 8 meses), detectados por detect_flowering_events.
    agg: "sum" (ej. precipitación acumulada) o "mean" (ej. NDVI promedio).
    """
    daily_df = daily_df.copy()
    daily_df[date_col] = pd.to_datetime(daily_df[date_col])
    resultados = []

    for _, ev in eventos.iterrows():
        muni = ev[muni_col]
        for etiqueta, fecha_centro in [("floracion", ev["fecha_floracion"]),
                                        ("cosecha", ev["fecha_cosecha_estimada"])]:
            ini = fecha_centro - pd.Timedelta(days=dias_ventana)
            fin = fecha_centro + pd.Timedelta(days=dias_ventana)
            mask = (daily_df[muni_col] == muni) & daily_df[date_col].between(ini, fin)
            valores = daily_df.loc[mask, value_col]
            if len(valores) == 0:
                continue
            resultados.append({
                muni_col: muni,
                "anio": ev["anio"],
                "evento": ev["evento"],
                f"{value_col}_{etiqueta}_dinamica": valores.sum() if agg == "sum" else valores.mean(),
            })

    return pd.DataFrame(resultados)
def phenology_features_por_seccional(df: pd.DataFrame, value_col: str,
                                      date_col: str = "fecha", muni_col: str = "municipio",
                                      agg: str = "sum") -> pd.DataFrame:
    """
    Igual que rolling_window_sum, pero usa la ventana de floración y de traviesa/cosecha
    ESPECÍFICA de la seccional de cada municipio, en vez de una sola ventana fija para
    los 57 municipios. Donde no hay seccional identificada, usa el respaldo genérico.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df["anio"] = df[date_col].dt.year
    df["mes"] = df[date_col].dt.month

    filas_largas = []
    for muni, g in df.groupby(muni_col):
        cal = get_calendario_municipio(muni)
        for etiqueta, rango in cal.items():
            if rango is None:
                continue
            m_ini, m_fin = rango
            sub = g[g["mes"].between(m_ini, m_fin)]
            agg_anual = sub.groupby("anio")[value_col].agg(agg).reset_index()
            agg_anual[muni_col] = muni
            agg_anual["ventana"] = etiqueta
            filas_largas.append(agg_anual)

    largo = pd.concat(filas_largas, ignore_index=True)
    ancho = largo.pivot_table(index=[muni_col, "anio"], columns="ventana", values=value_col)
    ancho.columns = [f"{value_col}_{c}_seccional" for c in ancho.columns]
    return ancho.reset_index()

def get_calendario_municipio(municipio: str) -> dict:
    """Devuelve el calendario fenológico (seccional si se conoce, genérico si no)."""
    seccional = MUNICIPIO_SECCIONAL.get(municipio, "Sin_seccional")
    return SECCIONAL_CALENDARIO.get(seccional, CALENDARIO_GENERICO)


# Comparacion de ventanas fenologicas

## Ventana fonologica general

In [40]:
# --- Ventanas fenológicas del café
# Cafeteros de Boyacá; estos rangos son orientativos para régimen bimodal andino) ---
VENTANAS_FENOLOGICAS = {
    "floracion_principal": (3, 4),       # marzo-abril
    "llenado_grano_principal": (6, 8),   # junio-agosto
    "cosecha_principal": (10, 12),       # octubre-diciembre
    "floracion_mitaca": (9, 10),         # floración secundaria
}

## Ventana fenologica dirigida por datos de precipitacion
la **floración del café se dispara por lluvia después de un período
seco**, no por el calendario — y que el momento varía según la altitud del municipio.

In [41]:
eventos_floracion = detect_flowering_events(chirps)
print(f"Eventos de floración detectados: {eventos_floracion.shape[0]}")
print(f"Promedio de eventos por municipio-año: "
      f"{eventos_floracion.shape[0] / (chirps['municipio'].nunique() * chirps['anio'].nunique()):.1f}")
eventos_floracion.head()

Eventos de floración detectados: 2006
Promedio de eventos por municipio-año: 1.9


,municipio,anio,evento,fecha_floracion,fecha_cosecha_estimada
0,Almeida,2007,1,2007-01-26,2007-09-26
1,Almeida,2007,2,2007-06-16,2008-02-16
2,Almeida,2008,1,2008-03-14,2008-11-14
3,Almeida,2008,2,2008-11-17,2009-07-17
4,Almeida,2009,1,2009-06-03,2010-02-03


In [42]:
precip_dinamica = phenology_window_features(
    chirps, eventos_floracion, value_col=precip_col, dias_ventana=20, agg="sum"
)

# Nos quedamos con el evento principal (evento==1) por municipio-año para no duplicar filas
precip_dinamica_principal = precip_dinamica[precip_dinamica["evento"] == 1].drop(columns="evento")

print(f"Features dinámicas (evento principal): {precip_dinamica_principal.shape}")
precip_dinamica_principal.head()

Features dinámicas (evento principal): (2106, 4)


,municipio,anio,precipitation_chirps_floracion_dinamica,precipitation_chirps_cosecha_dinamica
0,Almeida,2007,26.657409,NaN
1,Almeida,2007,NaN,208.350504
4,Almeida,2008,54.780009,NaN
5,Almeida,2008,NaN,258.279966
8,Almeida,2009,297.007749,NaN


## Ventana fenologica por seccional
| Seccional | Floración principal | Traviesa (cosecha menor) |
|---|---|---|
| Ricaurte | Oct-Dic | Abr-May |
| Lengupá | Oct-Dic | (no tiene) |
| Occidente | Abr-Jun | Sep-Oct |


In [43]:
MUNICIPIO_SECCIONAL = {
    # --- Ricaurte (72% de la producción de Boyacá) ---
    "Moniquira": "Ricaurte", "Togui": "Ricaurte", "San Jose De Pare": "Ricaurte",
    "Chitaraque": "Ricaurte", "Santana": "Ricaurte",
    # --- Lengupá (coincide con la provincia oficial completa) ---
    "Berbeo": "Lengupa", "Campohermoso": "Lengupa", "Miraflores": "Lengupa",
    "Paez": "Lengupa", "San Eduardo": "Lengupa", "Zetaquira": "Lengupa",
    # --- Occidente / La Libertad ---
    "Briceño": "Occidente", "Buenavista": "Occidente", "Coper": "Occidente",
    "La Victoria": "Occidente", "Maripi": "Occidente", "Muzo": "Occidente",
    "Otanche": "Occidente", "Pauna": "Occidente", "Quipama": "Occidente",
    "San Pablo De Borbur": "Occidente", "Santa Maria": "Occidente", "Tunungua": "Occidente",
    # --- Valle de Tenza (no tenemos calendario propio; usa respaldo genérico) ---
    "Almeida": "Valle_Tenza", "Chinavita": "Valle_Tenza", "Garagoa": "Valle_Tenza",
    "Guateque": "Valle_Tenza", "Guayata": "Valle_Tenza", "Macanal": "Valle_Tenza",
    "Pachavita": "Valle_Tenza", "Somondoco": "Valle_Tenza",
    # --- Cafetero confirmado, seccional exacta sin identificar ---
    "Labranzagrande": "Sin_seccional", "Pajarito": "Sin_seccional", "Paya": "Sin_seccional",
    "Pisba": "Sin_seccional", "Rondon": "Sin_seccional", "San Luis De Gaceno": "Sin_seccional",
}

In [44]:
SECCIONAL_CALENDARIO = {
    "Ricaurte": {"floracion_principal": (10, 12), "traviesa": (4, 5)},
    "Lengupa": {"floracion_principal": (10, 12), "traviesa": None},
    "Occidente": {"floracion_principal": (4, 6), "traviesa": (9, 10)},
}
# Respaldo genérico (Cenicafé nacional) para Valle de Tenza y municipios sin clasificar,
# donde no tenemos calendario específico por seccional.
CALENDARIO_GENERICO = {"floracion_principal": (3, 4), "traviesa": None}

In [45]:
precip_seccional = phenology_features_por_seccional(chirps, value_col=precip_col, agg="sum")

# Cuántos municipios caen en cada seccional (y cuántos usan el respaldo genérico)
conteo_seccional = pd.Series(MUNICIPIO_SECCIONAL).value_counts()
sin_clasificar = set(chirps["municipio"].unique()) - set(MUNICIPIO_SECCIONAL.keys())
print("Municipios por seccional:")
print(conteo_seccional)
print(f"\nSin seccional identificada (usan calendario genérico): {len(sin_clasificar)}")

precip_seccional.head()

Municipios por seccional:
Occidente        12
Valle_Tenza       8
Lengupa           6
Sin_seccional     6
Ricaurte          5
Name: count, dtype: int64

Sin seccional identificada (usan calendario genérico): 20


,municipio,anio,precipitation_chirps_floracion_principal_seccional,precipitation_chirps_traviesa_seccional
0,Almeida,2007,378.665886,NaN
1,Almeida,2008,177.223701,NaN
2,Almeida,2009,283.384658,NaN
3,Almeida,2010,437.501017,NaN
4,Almeida,2011,402.444936,NaN


## Comparacion

# Recalculo sobre CHIRPS

In [46]:
chirps = pd.read_csv(CLEAN_DIR / "CHIRPS_clean.csv")
chirps["fecha"] = pd.to_datetime(chirps["fecha"])
chirps["anio"] = chirps["fecha"].dt.year
precip_col = "precipitation_chirps"

# --- Enfoque 1: fijo por calendario ---
fijo = None
for nombre, (m_ini, m_fin) in VENTANAS_FENOLOGICAS.items():
    v = rolling_window_sum(chirps, value_col=precip_col, window_days=None,
                            window_start_month=m_ini, window_end_month=m_fin,
                            out_col_name=f"precip_{nombre}_FIJO")
    fijo = v if fijo is None else fijo.merge(v, on=["municipio", "anio"], how="outer")

# --- Enfoque 2: dinámico por evento de lluvia ---
eventos = detect_flowering_events(chirps)
dinamico_raw = phenology_window_features(chirps, eventos, value_col=precip_col,
                                          dias_ventana=20, agg="sum")
dinamico = dinamico_raw[dinamico_raw["evento"] == 1].drop(columns="evento")
dinamico = dinamico.rename(columns={
    f"{precip_col}_floracion_dinamica": "precip_floracion_DINAMICO",
    f"{precip_col}_cosecha_dinamica": "precip_cosecha_DINAMICO",
})

# --- Enfoque 3: por seccional ---
seccional = phenology_features_por_seccional(chirps, value_col=precip_col, agg="sum")
seccional = seccional.rename(columns={
    f"{precip_col}_floracion_principal_seccional": "precip_floracion_SECCIONAL",
    f"{precip_col}_traviesa_seccional": "precip_traviesa_SECCIONAL",
})

print("Fijo:", fijo.shape, "| Dinámico:", dinamico.shape, "| Seccional:", seccional.shape)

Fijo: (1083, 6) | Dinámico: (2106, 4) | Seccional: (1083, 4)


In [47]:
target = pd.read_csv(CLEAN_DIR / "Agronet_target_clean.csv")

comparacion = target[["municipio", "anio", "rendimiento_t_ha"]].copy()
comparacion = comparacion.merge(fijo, on=["municipio", "anio"], how="left")
comparacion = comparacion.merge(dinamico, on=["municipio", "anio"], how="left")
comparacion = comparacion.merge(seccional, on=["municipio", "anio"], how="left")

cols_features = [c for c in comparacion.columns if c not in ["municipio", "anio", "rendimiento_t_ha"]]

resultados = []
for col in cols_features:
    sub = comparacion[[col, "rendimiento_t_ha"]].dropna()
    if len(sub) < 30:
        resultados.append({"feature": col, "n_obs": len(sub), "correlacion": np.nan, "r2_holdout": np.nan})
        continue
    corr = sub[col].corr(sub["rendimiento_t_ha"])

    X = sub[[col]].values
    y = sub["rendimiento_t_ha"].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    modelo = LinearRegression().fit(X_train, y_train)
    r2 = r2_score(y_test, modelo.predict(X_test))

    resultados.append({"feature": col, "n_obs": len(sub), "correlacion": corr, "r2_holdout": r2})

resumen = pd.DataFrame(resultados).sort_values("r2_holdout", ascending=False)
resumen

,feature,n_obs,correlacion,r2_holdout
0,precip_floracion_principal_FIJO,1620,-0.054260,-0.000633
6,precip_floracion_SECCIONAL,1620,0.027252,-0.000965
2,precip_cosecha_principal_FIJO,1620,0.048905,-0.001371
5,precip_cosecha_DINAMICO,793,0.004849,-0.003000
1,precip_llenado_grano_principal_FIJO,1620,-0.007851,-0.003091
3,precip_floracion_mitaca_FIJO,1620,0.039378,-0.008081
7,precip_traviesa_SECCIONAL,619,-0.007557,-0.013669
4,precip_floracion_DINAMICO,806,0.064263,-0.027453


In [48]:
resumen["enfoque"] = resumen["feature"].str.extract(r"_(FIJO|DINAMICO|SECCIONAL)$")

veredicto = resumen.groupby("enfoque").agg(
    correlacion_abs_promedio=("correlacion", lambda x: x.abs().mean()),
    r2_promedio=("r2_holdout", "mean"),
    n_variables=("feature", "count"),
).sort_values("r2_promedio", ascending=False)

print("VEREDICTO por enfoque de ventana fenológica:")
veredicto

VEREDICTO por enfoque de ventana fenológica:


,correlacion_abs_promedio,r2_promedio,n_variables
enfoque,,,
FIJO,0.037599,-0.003294,4
SECCIONAL,0.017405,-0.007317,2
DINAMICO,0.034556,-0.015227,2


In [ ]:
def a_anomalia(df, col, muni_col="municipio"):
    media = df.groupby(muni_col)[col].transform("mean")
    return df[col] - media

comparacion_anomalia = comparacion.copy()
comparacion_anomalia["rendimiento_anomalia"] = a_anomalia(comparacion_anomalia, "rendimiento_t_ha")
for col in cols_features:
    comparacion_anomalia[f"{col}_anom"] = a_anomalia(comparacion_anomalia.dropna(subset=[col]).reindex(comparacion_anomalia.index), col)

resultados_anom = []
for col in cols_features:
    col_anom = f"{col}_anom"
    sub = comparacion_anomalia[[col_anom, "rendimiento_anomalia"]].dropna()
    if len(sub) < 30:
        continue
    corr = sub[col_anom].corr(sub["rendimiento_anomalia"])
    X, y = sub[[col_anom]].values, sub["rendimiento_anomalia"].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    modelo = LinearRegression().fit(X_train, y_train)
    r2 = r2_score(y_test, modelo.predict(X_test))
    resultados_anom.append({"feature": col, "n_obs": len(sub), "correlacion_anomalia": corr, "r2_holdout_anomalia": r2})

resumen_anom = pd.DataFrame(resultados_anom).sort_values("r2_holdout_anomalia", ascending=False)
resumen_anom["enfoque"] = resumen_anom["feature"].str.extract(r"_(FIJO|DINAMICO|SECCIONAL)$")
print(resumen_anom)
print()
veredicto_anom = resumen_anom.groupby("enfoque").agg(
    correlacion_abs_promedio=("correlacion_anomalia", lambda x: x.abs().mean()),
    r2_promedio=("r2_holdout_anomalia", "mean"),
).sort_values("r2_promedio", ascending=False)
print("VEREDICTO (con anomalías intra-municipales):")
veredicto_anom

El enfoque de Ventana fenologica fija sera el usado ya que se cubren todos los municipios, los datos seccionales no lo hacian y los datos dinamicos de la precipitacion no se presentan en los municipios mas secos.  

## Nuevas variables + Cambio de granularidad

## CHIRPS

In [49]:
chirps = pd.read_csv(CLEAN_DIR / "CHIRPS_clean.csv")
chirps["fecha"] = pd.to_datetime(chirps["fecha"])
precip_col = "precipitation_chirps"

# 1) Acumulado anual total
chirps["anio"] = chirps["fecha"].dt.year
chirps_annual = chirps.groupby(["municipio", "anio"])[precip_col].sum().reset_index()
chirps_annual = chirps_annual.rename(columns={precip_col: "precip_anual_mm"})

# 2) Rachas secas / días lluviosos — variable pedida explícitamente
dry = dry_spell_features(chirps, precip_col=precip_col)

# 3) Precipitación acumulada por ventana fenológica
ventanas_precip = []
for nombre, (m_ini, m_fin) in VENTANAS_FENOLOGICAS.items():
    v = rolling_window_sum(
        chirps, value_col=precip_col, window_days=None,
        window_start_month=m_ini, window_end_month=m_fin,
        out_col_name=f"precip_{nombre}_mm",
    )
    ventanas_precip.append(v)

chirps_features = chirps_annual.merge(dry, on=["municipio", "anio"], how="left")
for v in ventanas_precip:
    chirps_features = chirps_features.merge(v, on=["municipio", "anio"], how="left")

# 4) Anomalía de precipitación anual (respecto a la historia del propio municipio)
chirps_features = historical_anomaly(chirps_features, value_col="precip_anual_mm")

print(f"CHIRPS features: {chirps_features.shape}")
chirps_features.head()

CHIRPS features: (1083, 14)


,municipio,anio,precip_anual_mm,dias_secos_consecutivos_max,dias_secos_totales,dias_lluviosos_totales,precip_dia_lluvioso_promedio,precip_floracion_principal_mm,precip_llenado_grano_principal_mm,precip_cosecha_principal_mm,precip_floracion_mitaca_mm,precip_anual_mm_media_hist,precip_anual_mm_std_hist,precip_anual_mm_anomalia
0,Almeida,2007,1970.405330,25,249,116,16.945981,378.665886,720.148873,360.374687,328.122151,1842.506745,277.504534,0.460888
1,Almeida,2008,2145.819213,17,237,129,16.619268,177.223701,987.202533,438.512397,384.569097,1842.506745,277.504534,1.093000
2,Almeida,2009,1763.771071,32,246,119,14.772656,283.384658,799.729197,286.811540,287.128496,1842.506745,277.504534,-0.283728
3,Almeida,2010,2131.727295,14,207,158,13.468566,437.501017,720.067184,485.250050,288.755197,1842.506745,277.504534,1.042219
4,Almeida,2011,2265.725688,16,222,143,15.823820,402.444936,700.750462,521.650448,450.121992,1842.506745,277.504534,1.525088


## ERA 5

In [50]:
era5 = pd.read_csv(CLEAN_DIR / "ERA5_clean.csv")
era5["fecha"] = pd.to_datetime(era5["fecha"])
era5["anio"] = era5["fecha"].dt.year

# 1) Agregados anuales básicos
era5_basic = era5.groupby(["municipio", "anio"]).agg(
    temp_media_anual=("temp_mean", "mean"),
    temp_min_anual=("temp_min", "min"),
    temp_max_anual=("temp_max", "max"),
    temp_std_anual=("temp_mean", "std"),
    humedad_relativa_media=("relative_humidity", "mean"),
    soil_moisture_media=("soil_moisture", "mean"),
    soil_moisture_min=("soil_moisture", "min"),
    solar_radiation_media=("solar_radiation", "mean"),
    evaporacion_total=("evaporation", "sum"),
    precip_era5_anual=("precipitation", "sum"),
).reset_index()

# 2) Días de frío / calor extremo (estrés térmico)
stress = frost_and_heat_days(era5, tmin_col="temp_min", tmax_col="temp_max")

# 3) Growing Degree Days
gdd = growing_degree_days(era5, tmax_col="temp_max", tmin_col="temp_min")

era5_features = era5_basic.merge(stress, on=["municipio", "anio"], how="left")
era5_features = era5_features.merge(gdd, on=["municipio", "anio"], how="left")

# 4) Anomalía de temperatura respecto a la historia del propio municipio
era5_features = historical_anomaly(era5_features, value_col="temp_media_anual")

print(f"ERA5 features: {era5_features.shape}")
era5_features.head()

ERA5 features: (1083, 18)


,municipio,anio,temp_media_anual,temp_min_anual,temp_max_anual,temp_std_anual,humedad_relativa_media,soil_moisture_media,soil_moisture_min,solar_radiation_media,evaporacion_total,precip_era5_anual,dias_frio,dias_calor_extremo,gdd_anual,temp_media_anual_media_hist,temp_media_anual_std_hist,temp_media_anual_anomalia
0,Almeida,2007,15.364114,5.801661,24.123140,0.827625,77.947978,0.298301,0.223098,18.059856,-885.345654,930.823010,5,0,2090.112673,15.332776,0.290707,0.107799
1,Almeida,2008,15.034777,8.178275,22.505538,0.642944,81.685336,0.335925,0.267446,17.327535,-998.194210,1300.411690,0,0,1958.147656,15.332776,0.290707,-1.025083
2,Almeida,2009,15.388480,8.770089,22.899561,0.664381,79.825869,0.328801,0.247406,17.637935,-992.575067,936.341676,0,0,2091.689382,15.332776,0.290707,0.191614
3,Almeida,2010,15.804438,5.185118,24.735151,1.131739,79.743288,0.342607,0.218450,17.546276,-898.527976,1524.927710,4,0,2256.436565,15.332776,0.290707,1.622465
4,Almeida,2011,15.049476,8.363143,21.419075,0.501823,84.055159,0.383890,0.296903,16.641601,-1021.318764,1902.128412,0,0,1977.104243,15.332776,0.290707,-0.974521


## TerraClimate

In [51]:
tc = pd.read_csv(CLEAN_DIR / "TerraClimate_clean.csv")
tc["anio"] = tc["fecha"].str[:4].astype(int)

tc_features = tc.groupby(["municipio", "anio"]).agg(
    pr_anual_mm=("pr", "sum"),
    tmmx_media_anual=("tmmx", "mean"),
    tmmn_media_anual=("tmmn", "mean"),
    aet_anual_mm=("aet", "sum"),
    pet_anual_mm=("pet", "sum"),
    def_anual_mm=("def", "sum"),
    soil_media_anual=("soil", "mean"),
    ro_anual_mm=("ro", "sum"),
    srad_media_anual=("srad", "mean"),
    vpd_media_anual=("vpd", "mean"),
    pdsi_media_anual=("pdsi", "mean"),
    pdsi_min_anual=("pdsi", "min"),
).reset_index()

# Balance hídrico anual: variable derivada muy usada en índices paramétricos agrícolas
tc_features["balance_hidrico_anual"] = tc_features["pr_anual_mm"] - tc_features["pet_anual_mm"]

# Anomalías
tc_features = historical_anomaly(tc_features, value_col="pr_anual_mm")
tc_features = historical_anomaly(tc_features, value_col="balance_hidrico_anual")

print(f"TerraClimate features: {tc_features.shape}")
tc_features.head()

TerraClimate features: (1026, 21)


,municipio,anio,pr_anual_mm,tmmx_media_anual,tmmn_media_anual,aet_anual_mm,pet_anual_mm,def_anual_mm,soil_media_anual,ro_anual_mm,srad_media_anual,vpd_media_anual,pdsi_media_anual,pdsi_min_anual,balance_hidrico_anual,pr_anual_mm_media_hist,pr_anual_mm_std_hist,pr_anual_mm_anomalia,balance_hidrico_anual_media_hist,balance_hidrico_anual_std_hist,balance_hidrico_anual_anomalia
0,Almeida,2007,2036.121739,19.234101,11.662754,956.144696,1038.946957,82.823826,209.416638,107.014957,182.085058,0.434833,2.025488,1.254209,997.174783,2021.056425,414.821623,0.036318,977.170213,424.821714,0.047089
1,Almeida,2008,2589.182609,19.715899,10.621797,958.414609,997.998957,39.599652,216.690565,162.835826,170.397971,0.468607,4.202830,1.783461,1591.183652,2021.056425,414.821623,1.369567,977.170213,424.821714,1.445344
2,Almeida,2009,1981.022609,20.555638,10.916362,1026.947304,1069.803304,42.813391,221.309696,98.851304,183.650464,0.491119,4.633839,1.706122,911.219304,2021.056425,414.821623,-0.096509,977.170213,424.821714,-0.155244
3,Almeida,2010,2534.206957,20.777232,11.289826,958.014957,1042.727478,84.675652,202.100159,153.785391,176.288565,0.497706,1.779913,-2.213252,1491.479478,2021.056425,414.821623,1.237039,977.170213,424.821714,1.210647
4,Almeida,2011,2844.088696,20.190522,11.168333,987.505391,1006.753739,19.283130,224.494319,185.684174,171.099478,0.462839,8.276535,6.000504,1837.334957,2021.056425,414.821623,1.984063,977.170213,424.821714,2.024766


## MODIS

In [52]:
modis = pd.read_csv(CLEAN_DIR / "MODIS_clean.csv")
modis["fecha"] = pd.to_datetime(modis["fecha"])
modis["anio"] = modis["fecha"].dt.year

VENTANAS_MODIS = {
    "floracion_principal": (3, 4),
    "llenado_grano_principal": (6, 8),
}

modis_basic = modis.groupby(["municipio", "anio"]).agg(
    ndvi_media_anual=("NDVI", "mean"),
    ndvi_min_anual=("NDVI", "min"),
    ndvi_std_anual=("NDVI", "std"),
    evi_media_anual=("EVI", "mean"),
    evi_min_anual=("EVI", "min"),
    calidad_media_anual=("SummaryQA", "mean"),
).reset_index()

ventanas_modis = []
for nombre, (m_ini, m_fin) in VENTANAS_MODIS.items():
    mask = modis["fecha"].dt.month.between(m_ini, m_fin)
    sub = modis[mask]
    v = sub.groupby(["municipio", "anio"]).agg(**{
        f"ndvi_{nombre}": ("NDVI", "mean"),
        f"evi_{nombre}": ("EVI", "mean"),
    }).reset_index()
    ventanas_modis.append(v)

modis_features = modis_basic
for v in ventanas_modis:
    modis_features = modis_features.merge(v, on=["municipio", "anio"], how="left")

modis_features = historical_anomaly(modis_features, value_col="ndvi_media_anual")

print(f"MODIS features: {modis_features.shape}")
modis_features.head()

MODIS features: (1083, 15)


,municipio,anio,ndvi_media_anual,ndvi_min_anual,ndvi_std_anual,evi_media_anual,evi_min_anual,calidad_media_anual,ndvi_floracion_principal,evi_floracion_principal,ndvi_llenado_grano_principal,evi_llenado_grano_principal,ndvi_media_anual_media_hist,ndvi_media_anual_std_hist,ndvi_media_anual_anomalia
0,Almeida,2007,0.457887,0.140208,0.200823,0.296345,0.110820,2.142047,0.275288,0.192523,0.344452,0.244987,0.478672,0.041803,-0.497208
1,Almeida,2008,0.447700,0.108067,0.183571,0.295801,0.095823,2.217180,0.380241,0.252221,0.319074,0.232276,0.478672,0.041803,-0.740911
2,Almeida,2009,0.464793,0.102689,0.200901,0.311535,0.090552,2.140706,0.339795,0.241093,0.298141,0.219388,0.478672,0.041803,-0.332001
3,Almeida,2010,0.529342,0.213081,0.154149,0.345011,0.179218,1.984371,0.491509,0.333122,0.397740,0.292215,0.478672,0.041803,1.212128
4,Almeida,2011,0.531551,0.201314,0.169614,0.345566,0.152490,2.015090,0.410188,0.272119,0.427548,0.292696,0.478672,0.041803,1.264951


# Datos Anuales - Municipales

In [53]:
srtm = pd.read_csv(CLEAN_DIR / "SRTM_clean.csv")  # estático, se une solo por municipio

panel = chirps_features.merge(era5_features, on=["municipio", "anio"], how="outer")
panel = panel.merge(tc_features, on=["municipio", "anio"], how="outer")
panel = panel.merge(modis_features, on=["municipio", "anio"], how="outer")
panel = panel.merge(srtm, on="municipio", how="left")

# Reporte de completitud (requerimiento funcional F2: panel >= 90% completo)
completitud = 1 - panel.isnull().mean()
print("Completitud por columna (10 con más faltantes):")
print(completitud.sort_values().head(10))
print(f"\nCompletitud global del panel: {completitud.mean():.1%}")
print(f"Filas: {panel.shape[0]} | Columnas: {panel.shape[1]}")
print(f"Municipios: {panel['municipio'].nunique()} | Años: {sorted(panel['anio'].unique())}")

Completitud por columna (10 con más faltantes):
tmmn_media_anual    0.947368
tmmx_media_anual    0.947368
pr_anual_mm         0.947368
def_anual_mm        0.947368
soil_media_anual    0.947368
ro_anual_mm         0.947368
srad_media_anual    0.947368
vpd_media_anual     0.947368
pdsi_media_anual    0.947368
pdsi_min_anual      0.947368
dtype: float64

Completitud global del panel: 98.5%
Filas: 1083 | Columnas: 66
Municipios: 57 | Años: [np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


In [54]:
panel.to_csv(CLEAN_DIR / "panel_anual_municipal.csv", index=False)
print("Guardado: panel_anual_municipal.csv")

Guardado: panel_anual_municipal.csv


# Validacion de datos productivos (Agronet) con datos de vegetacion NDVI

In [55]:
target = pd.read_csv(CLEAN_DIR / "Agronet_target_clean.csv")

panel_final = panel.merge(
    target[["municipio", "anio", "rendimiento_t_ha", "area_cosechada_ha", "produccion_ton"]],
    on=["municipio", "anio"], how="inner",  # inner: solo años/municipios con target real
)

print(f"Panel final con target: {panel_final.shape[0]} filas de {panel.shape[0]} en el panel de features")
panel_final.to_csv(CLEAN_DIR / "panel_final_con_target.csv", index=False)
print("Guardado: panel_final_con_target.csv")

Panel final con target: 827 filas de 1083 en el panel de features
Guardado: panel_final_con_target.csv


In [56]:
corr_global = panel_final["ndvi_media_anual"].corr(panel_final["rendimiento_t_ha"])
print(f"Correlación global NDVI medio anual vs. rendimiento: {corr_global:.3f}")

corr_por_municipio = panel_final.groupby("municipio").apply(
    lambda g: g["ndvi_media_anual"].corr(g["rendimiento_t_ha"]) if len(g) >= 4 else np.nan,
    include_groups=False,
).sort_values()

print("\nMunicipios con correlación NDVI-rendimiento más baja/negativa "
      "(candidatos a revisar extracción GEE):")
corr_por_municipio.head(10)

Correlación global NDVI medio anual vs. rendimiento: 0.088

Municipios con correlación NDVI-rendimiento más baja/negativa (candidatos a revisar extracción GEE):


municipio
Guacamayas           -0.640933
Santa Sofia          -0.566172
Socota               -0.466814
Sutatenza            -0.290297
Macanal              -0.288200
La Capilla           -0.282331
San Mateo            -0.258185
Tenza                -0.243356
Chivor               -0.229890
San Luis De Gaceno   -0.213950
dtype: float64